# Letting the LLM read the SysML

This project used to hand-parse SysML v2. `sysml/pipeline/parse.py` was 791 lines of
grammar -- a scanner, a declaration splitter, a reference resolver -- and it produced
`out/model.json`, which two more steps turned into a graph. That is gone. The graph is
now built by **graphrag_importer's own extraction pipeline**, the same code the
platform's importer pods run, called in-process against a local ArangoDB.

This notebook is about what changed and what it bought. `simple-demo.ipynb` is about
asking the resulting graph questions, and it needed almost no changes -- which is
itself part of the point.

```bash
docker run -d --name christian-webb-drone-arango -p 8529:8529 \
  -e ARANGO_ROOT_PASSWORD=testpass arangodb:3.12.9.4 \
  arangod --experimental-vector-index=true
python build.py
```

The OpenAI key comes from `CHAT_API_KEY` (or `OPENAI_API_KEY`) in the `env` file one
directory up; an exported variable is used only if the file has neither.

## The whole build

Two steps. `extract` runs the LLM over the source text and leaves a NetworkX graph
and some JSON in `out/kg`; `load` hands those files to the importer's own ArangoDB
writer. Neither knows anything about SysML.

In [1]:
import inspect, logging
from sysml import config, nl
from sysml.pipeline import analogy, extract, load

logging.disable(logging.INFO)  # the services narrate every step; keep just the answers

print(inspect.getsource(extract.graphrag))

def graphrag(model: str):
    """The extraction pipeline, configured and pointed at one model's workbench.

    `enable_chunk_embeddings` is on because the `unified` retriever searches the
    source chunks in parallel with the entity graph, and without vectors on the
    chunks that half of it has nothing to search.

    A workbench per model is what keeps entity names from merging across two
    unrelated vehicles -- the merge happens here, in the builder's own graph, long
    before ArangoDB sees anything, so it is the only place it can be scoped.

    The LLM cache lives in the working directory, so a second run over unchanged
    files re-reads the answers instead of re-buying them. It is keyed on the whole
    prompt (`_llm.py:71`), so changing anything in `prompts` invalidates it and the
    next run pays for the answers again.

    `entity_extract_max_gleaning=0` switches off the second pass over each chunk.
    It sends "MANY entities were missed in the last extraction ... loo

In [2]:
print(inspect.getsource(load.load))

async def load() -> dict:
    db = config.db(create=True)
    reset(db)

    # One import per model, in MODEL_NAMES order so the import numbers are the
    # ones `config.import_number` promises. Each reads its own workbench and
    # writes the same collections; keys cannot collide because the number is in
    # them.
    imp = None
    for model in config.MODEL_NAMES:
        imp = importer(model)
        await imp.initialize(config.token())
        await imp.import_documents(config.ARTIFACTS.FULL_DOCS)
        # Deliberately without the chunk-embedding file -- see `chunk_vectors`.
        await imp.import_text_chunks(config.ARTIFACTS.TEXT_CHUNKS)
        chunk_vectors(db, imp)
        await imp.import_entities(config.ARTIFACTS.ENTITIES)
        await imp.import_relationships(config.ARTIFACTS.RELATIONSHIPS)
        await imp.import_community_reports(config.ARTIFACTS.COMMUNITY_REPORTS)

    label(db)
    # Before the indexes, not after: `structure` creates an entity for anything a
   

That is the whole of it. `GraphRAG` and `ImportGraphToADB` are imported unmodified;
the only local code is the two `await` sequences above, plus a status-sink shim
because the writer announces its progress to a platform service that is not here.

## The ontology is the only thing we tell it about SysML

`entity_types` and `relationship_types` are constructor arguments. With
`enable_strict_types=True` they are a closed vocabulary rather than a hint: an
entity or edge whose type is not on the list is **dropped**, not renamed.

These two lists are exactly what the parser used to recognise in its grammar. The
same 27 declaration kinds and the same 18 relations -- moved out of Python and into
the prompt.

In [3]:
print(f"{len(config.KINDS)} entity types")
print("  " + ", ".join(config.KINDS))
print(f"\n{len(config.RELATIONS_ONTOLOGY)} relation types")
print("  " + ", ".join(config.RELATIONS_ONTOLOGY))

27 entity types
  Package, Part, Action, State, Port, Item, Attribute, Requirement, Calc, Analysis, Connection, Interface, View, Viewpoint, Enumeration, Concern, Constraint, Flow, Allocation, Event, Metadata, UseCase, Rendering, Verification, Snapshot, Timeslice, Occurrence

18 relation types
  owns, typedBy, specializes, redefines, satisfies, refines, derives, performs, subject, exhibits, connects, transitionsTo, variantOf, imports, sliceOf, sends, dependsOn, valueRef


## What came out

The database is built by `python build.py`. This connects to it.

In [4]:
db = config.db()
for name in config.ALL_COLLECTIONS:
    print(f"{db.collection(name).count():>6}  {name}")

    30  sysml_Documents
   114  sysml_Chunks
  2291  sysml_Entities
   280  sysml_Communities
 11982  sysml_Relations


## 1. The ontology held

Every relation type the extraction produced, counted. All of them are from the list
above -- `enable_strict_types` guarantees it, and the last cell checks rather than
assumes.

`owns` dominates because containment is what these files are mostly made of, and
both halves report it: the lexer for every declaration, extraction for the ones it
recognises in the text. `refines` is the interesting one at 244, because it relates
two *statements* rather than two parts and no SysML keyword spells it out -- the
lexer cannot produce a single one of those.

In [5]:
Q = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT kind = r.relationship_type WITH COUNT INTO n
  SORT n DESC RETURN {{kind, n}}'''
rows = list(db.aql.execute(Q))
for row in rows:
    print(f"{row['n']:>6}  {row['kind']}")

off = [r["kind"] for r in rows if r["kind"] not in config.RELATIONSHIP_TYPES]
print(f"\n{len(rows)} of {len(config.RELATIONS_ONTOLOGY)} offered types used, "
      f"{len(off)} off-ontology: {off or 'none'}")

  1793  owns
   556  typedby
   486  specializes
   422  satisfies
   244  refines
   104  performs
    29  imports
    23  redefines
    16  connects
    14  transitionsto
     5  subject
     2  dependson

12 of 18 offered types used, 0 off-ontology: none


## 2. It reads the prose, not just the syntax

`dependsOn` is in our ontology but there is no SysML keyword for it. The parser
could only ever emit a relation that some statement spelled out. These edges exist
because the text explains a dependency in words.

In [6]:
Q = f'''
FOR r IN {config.RELATIONS}
  FILTER r.relationship_type == "dependson"
  LIMIT 4
  RETURN {{a: DOCUMENT(r._from).entity_name, b: DOCUMENT(r._to).entity_name,
          why: r.description}}'''
for row in db.aql.execute(Q):
    print(f"{row['a']}  ->  {row['b']}")
    print(f"    {row['why']}\n")

CALCULATEDELTAV  ->  CALCULATELAUNCHSTAGEDELTAV
    calculateDeltaV(isp, initialMass, finalMass)

CALCULATEDELTAV  ->  CALCULATESPACECRAFTBURNDELTAV
    calculateDeltaV(isp, initialMass, finalMass)



## 3. The same element in two files is one element

The parser keyed elements on their qualified name inside one file, so a requirement
declared in one package and referred to from another became two unrelated rows.
Extraction merges by name across the whole corpus, so an element accumulates every
file it appears in -- and the answer to "where is this discussed" stops being "one
place".

In [7]:
# Packages are excluded: the standard-library ones are imported by everything, so
# they crowd out the elements the question is actually about.
Q = f'''
FOR e IN {config.ENTITIES}
  FILTER LENGTH(e.files) > 1 AND e.entity_type != "package"
  SORT LENGTH(e.files) DESC LIMIT 6
  RETURN {{name: e.entity_name, type: e.entity_type, files: LENGTH(e.files)}}'''
total = next(iter(db.aql.execute(
    f'''RETURN LENGTH(FOR e IN {config.ENTITIES}
         FILTER LENGTH(e.files) > 1 AND e.entity_type != "package" RETURN 1)''')))
print(f"{total} non-package elements appear in more than one source file. The widest:")
for row in db.aql.execute(Q):
    print(f"  {row['files']:>3} files   {row['name']} ({row['type']})")

306 non-package elements appear in more than one source file. The widest:
    4 files   CREW (part)
    4 files   SPACECRAFT (part)
    4 files   PLANETARYPROTECTION (part)
    4 files   APOLLO11MISSIONSYSTEM (part)
    4 files   COMMANDMODULE (part)
    3 files   PLSS (part)


Across model boundaries it deliberately does **not** happen. Each model is extracted
into its own workbench and imported under its own `import_number`, which the writer
puts in every document key, so two models that use the same word are two rows. The
query below is the check, and it is meant to come back empty: a `Battery` in one
vehicle is not a `Battery` in another, and merging them would silently answer a
question about one vehicle with the other's numbers.

What relates two models is the analogy layer, and it says *resembles*, not *is* --
`analogy-demo.ipynb`.

In [8]:
Q = f'''
FOR e IN {config.ENTITIES}
  FILTER LENGTH(e.models) > 1
  RETURN {{name: e.entity_name, type: e.entity_type, models: e.models}}'''
shared = list(db.aql.execute(Q))
for row in shared:
    print(f"  {row['name']:<16} {row['type']:<12} {', '.join(row['models'])}")
print(f"{len(shared)} elements belong to more than one model.")

# The same word in two models, kept apart by the import number in the key.
Q = f'''
FOR e IN {config.ENTITIES}
  FILTER e.entity_name == "BATTERY"
  RETURN {{key: e._key, models: e.models, at: e.source_file}}'''
for row in db.aql.execute(Q):
    print(f"  {row['key']:<24} {row['models'][0]:<24} {row['at']}")

0 elements belong to more than one model.


  7294977362189889090_1    Drone_BaseArchitecture   None
  7294977362189889090_0    DroneModelLogical        DroneModelLogical.sysml


## 4. Communities and their reports come with it

The old `enrich` step hand-rolled label propagation over the traceability edges,
picked a title from the largest member, and wrote its own report prompt -- about 200
lines. Leiden and the report writer are part of the extraction pipeline, so all of
that is now upstream's. Three levels of hierarchy, and the reports are structured
rather than a blob of text.

In [9]:
Q = f'''
FOR c IN {config.COMMUNITIES}
  FILTER c.level == 0
  SORT c.occurrence DESC LIMIT 1
  RETURN c'''
c = next(iter(db.aql.execute(Q)))
print(f"level {c['level']}   covers {c['occurrence']:.0%} of the corpus   "
      f"{len(c['sub_communities'])} sub-communities\n")
print(c["report_json"]["title"])
print(c["report_json"]["summary"][:400], "...\n")
for f in c["report_json"]["findings"][:2]:
    print(f"- {f['summary'] if isinstance(f, dict) else f}")

level 0   covers 100% of the corpus   14 sub-communities

Function: Densely Connected Group Including 'FUNCTION', 'APOLLOSPACECRAFT', and Related Actions
This group focuses on the 'FUNCTION' element, an abstract action definition with subfunctions typed by Function. It contains various 'action' elements and the 'part' element 'SPACECRAFT', typed by ApolloSpacecraft. The group also encompasses numerous specialized functions which detail actions related to spacecraft operations, such as 'SetTranslunarCourse' and 'ExploreLunarSurface'. The 'SPACECRAFT ...

- The 'FUNCTION' element is central to the group's cohesion.
- The 'SPACECRAFT' part is extensively integrated into mission steps.


## What it costs, and what is done about it

Nothing here is free.

| | parser | extraction |
|---|---|---|
| element names | `SaturnV`, as declared | `SATURNV` -- upper-cased at `_op.py:264` |
| `dryMass = 137000 [kg]` | a typed field you can sum in AQL | a number inside a sentence |
| `part stage1 : 'S-IC'` | an `owns` and a `typedBy` edge | reported, but not reliably |
| rebuild | free and identical | cached, but non-deterministic on a change |

The upper-casing is a real loss and nothing recovers it. The rest mattered too much
to leave: an LLM asked to read a declarative language reports what the text
*discusses*, and the two things it is worst at -- exact values and exact containment
-- are exactly what an analytical question needs.

How badly depends almost entirely on what it is asked. Upstream's prompt is
Microsoft GraphRAG's, whose three worked examples are all narrative fiction; on
these files it produced 68 `owns` edges where the syntax states about seventeen
hundred, and left `S-IC` with no relations of any kind. `sysml/pipeline/prompts.py`
replaces it with one that describes what a SysML declaration is and says to
transcribe rather than interpret, and turns off the gleaning pass that tells the
model it missed things. Same files, same model, same ontology:

| | upstream's prompt | this one |
|---|---|---|
| `owns` edges extraction found | 68 | 926 |
| `satisfy X by Y` statements found, of 265 | 99 | 219 |
| edges relating an element to itself | 12 satisfies, and more elsewhere | 0 satisfies, 54 in total |
| elements named but never declared | 1,286 | 440 |
| LLM calls per chunk | 3 | 1 |

So it is better, and it is still not the authority. `load` runs a second pass over
the same sources, `sysml/pipeline/structure.py`, which reads them with a lexer and
writes down only what the syntax states outright: attribute values, the containment
tree, typing, specialisation, and the `satisfy X by Y` statement. Where the two
disagree about structure, the lexer wins.

It is a smaller job than the parser it replaces, because it needs no name
resolution across files, no `doc` handling, and no notion of what any of the 27
declaration kinds *mean* -- only where each one is written and what it says.
`analytics-demo.ipynb` is what that buys.

In [10]:
Q = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT stated = r.stated == true WITH COUNT INTO n
  RETURN {{source: stated ? "read from the syntax" : "inferred by the LLM", n}}'''
for row in db.aql.execute(Q):
    print(f"{row['n']:>6}  {row['source']}")

n = next(iter(db.aql.execute(
    f"RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.attributes.dryMass != null RETURN 1)")))
print(f"\n{n} elements carry a dryMass the lexer read out of the file.")

   571  inferred by the LLM
  3123  read from the syntax



7 elements carry a dryMass the lexer read out of the file.


## The size of the change

The parser, the projection and the enrichment were 1,557 lines between them, all of
it ours to maintain and all of it specific to one input language. What replaced them
is about a third of that -- and only the last of the three files below knows what
SysML looks like.

In [11]:
from pathlib import Path

old = {"parse.py": 791, "project.py": 298, "enrich.py": 468}
new = {p.name: len(p.read_text(encoding="utf-8").splitlines())
       for p in [Path("sysml/pipeline/extract.py"), Path("sysml/pipeline/load.py"),
                 Path("sysml/pipeline/prompts.py"), Path("sysml/pipeline/structure.py")]}
print(f"removed  {sum(old.values()):>5}   " + ", ".join(f"{k} {v}" for k, v in old.items()))
print(f"added    {sum(new.values()):>5}   " + ", ".join(f"{k} {v}" for k, v in new.items()))
print(f"net      {sum(new.values()) - sum(old.values()):>5} lines")

removed   1557   parse.py 791, project.py 298, enrich.py 468
added     1601   extract.py 143, load.py 237, prompts.py 306, structure.py 915
net         44 lines


## What did not change

`sysml/nl.py`, the entire read side, kept working. The retrievers and the AQLizer
were pointed at a graph the importer's own writer produced, which is what they were
built to read -- where before they were pointed at a hand-made imitation of it.
`aql_examples.md` had to be rewritten, because the fields it teaches are different
ones, but no Python moved.

That is the argument for the change in one line: the graph is now produced by the
code that owns the schema, so agreeing with the schema is no longer something this
project has to keep doing by hand.

In [12]:
print((await nl.retriever().ask_async(
    "What is the drone battery for and what capacity does it have?")).answer[:700])

# Drone Battery Details

## Purpose of the Battery
The battery is a part of the drone's energy storage system. It plays a central role within the drone system, being declared as a component of the DRONE in both the DRONE and DRONE_SYSTEMREQUIREMENTS packages[CITE:1][CITE:2][CITE:3].

## Capacity of the Battery
The capacity of the drone's battery is specified as 6000. However, the model does not specify a unit for this capacity[CITE:1][CITE:3]. 

The capacity value plays a significant role in the MAXCAPACITY requirement, which asserts that the battery's capacity must be greater than or equal to 6000 to comply with system requirements[CITE:1]. 

## Note
While the battery's capacity is given as
